In [0]:
# imports
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [0]:
# silver class
class SilverTransformation:
    def __init__(self, catalog_name, bronze_schema_name, bronze_table_name, silver_schema_name, silver_table_name, silver_table_type, silver_table_mode, fillna_columns_datatypes, strictly_drop_columns):
        self.catalog_name = catalog_name
        self.bronze_schema_name = bronze_schema_name
        self.bronze_table_name = bronze_table_name
        self.silver_schema_name = silver_schema_name
        self.silver_table_name = silver_table_name
        self.silver_table_type = silver_table_type
        self.silver_table_mode = silver_table_mode
        self.columns_datatypes = fillna_columns_datatypes
        self.strictly_drop_columns = strictly_drop_columns

    def handle_missing_data(self):
        print("03: Missing data handling started")
        
        # for strict columns
        self.bronze_data = self.bronze_data.filter(
            F.col(self.strictly_drop_columns[0]).isNotNull() &
            F.col(self.strictly_drop_columns[1]).isNotNull() 
        )

        # for not-strict columns
        for column, datatype in self.columns_datatypes.items():
            if datatype == "string":
                self.bronze_data = self.bronze_data.fillna({column: "Unknown"})
            elif datatype in ["integer", "double"]:
                average = self.bronze_data.select(
                    F.avg(F.col(column)).alias("average")
                ).first()["average"]
                self.bronze_data = self.bronze_data.fillna({column: average})

        print("04: Missing data handling completed")

    def handle_duplicate_data(self):
        window_config = Window.partitionBy("transaction_id").orderBy("transaction_date")

        print("05: Handling Duplicate Data")

        self.bronze_data = self.bronze_data.withColumn(
            "duplicate_count",
            F.row_number().over(window_config)
        ).filter(
            F.col("duplicate_count") == 1
        ).drop(
            F.col("duplicate_count")
        )

        print("06: Duplicate data handling completed")

    def handle_future_dates(self):
        print("07: Handling Future Dates Data")
        self.bronze_data = self.bronze_data.filter(
            F.col("transaction_date") <= F.current_date()
        )
        print("08: Future Dates data handling completed")

    def transform_string_columns(self):
        print("09: Transforming string columns into lower case.")
        for column, datatype in self.columns_datatypes.items():
            if datatype == "string":
                self.bronze_data =  self.bronze_data.withColumn(
                    column,
                    F.trim(F.lower(F.col(column)))
                )

        print("10:  Transforming string columns into lower case completed.")

    def load_bronze_data(self):
        bronze_table_path = self.catalog_name + '.' + self.bronze_schema_name + '.' + self.bronze_table_name
        print(f"01: Bronze table: {bronze_table_path} loading started.")
        self.bronze_data = spark.read.table(bronze_table_path) 
        print(f"02: Bronze table: {bronze_table_path} Loaded.")

    def write_to_silver(self):
        print("11: Adding Silver Timestamp column")
        self.bronze_data = self.bronze_data.withColumn(
            "silver_ingestion_timestamp",
            F.current_timestamp()
        )
        print("12: Added Silver Timestamp column")

        silver_table_path = self.catalog_name + '.' + self.silver_schema_name + '.' + self.silver_table_name
        print(f"13: Writing data to silver table, path: {silver_table_path}")
        self.bronze_data.write.format(self.silver_table_type)\
                              .mode(self.silver_table_mode)\
                              .saveAsTable(silver_table_path)
        print(f"14: Writing data to silver table, path: {silver_table_path} is completed,")

In [0]:
fillna_columns = {
    "transaction_year": "integer",
    "transaction_quarter": "integer",
    "transaction_month": "integer",
    "product_name": "string",
    "product_category": "string",
    "quantity_unit": "string",
    "supplier_name": "string",
    "supplier_country": "string",
    "supplier_reliability_score": "double",
    "refinery_name": "string",
    "destination_city": "string",
    "transportation_mode": "string",
    "ordered_quantity": "double",
    "demand_quantity": "double",
    "available_inventory": "double",
    "unit_price_usd": "double",
    "product_cost_usd": "double",
    "transportation_cost_usd": "double",
    "total_cost_usd": "double",
    "expected_lead_time_days": "integer",
    "actual_lead_time_days": "integer",
    "delay_days": "integer",
    "is_delayed": "integer",
    "is_stockout": "integer",
    "quality_status": "string",
    "quality_score": "double",
    "delivery_status": "string",
}

strictly_drop_columns = ["transaction_id", "transaction_date"]

In [0]:
# main part
silver_transformation = SilverTransformation(
    catalog_name="supply_chain",
    bronze_schema_name="bronze",
    bronze_table_name="bronze_supply_chain",
    silver_schema_name="silver",
    silver_table_name="silver_supply_chain",
    silver_table_type="delta",
    silver_table_mode="overwrite",
    fillna_columns_datatypes=fillna_columns,
    strictly_drop_columns=strictly_drop_columns
)

silver_transformation.load_bronze_data()

silver_transformation.handle_missing_data()
silver_transformation.handle_duplicate_data()
silver_transformation.handle_future_dates()
silver_transformation.transform_string_columns()

silver_transformation.write_to_silver()

In [0]:
%sql
SELECT *
FROM supply_chain.silver.silver_supply_chain;

In [0]:
%sql
SELECT
    COUNT(*) AS rows_count
FROM supply_chain.silver.silver_supply_chain;